In [64]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [65]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [66]:
df = pd.read_csv('/content/drive/MyDrive/sahil nawaz alam/Emotion_classify_Data.csv')

In [67]:
df.head()

,Comment,Emotion
0,i seriously hate one subject to death but now ...,fear
1,im so full of life i feel appalled,anger
2,i sit here to write i start to dig out my feel...,fear
3,ive been really angry with r and i feel like a...,joy
4,i feel suspicious if there is no one outside l...,fear


In [68]:
df.Emotion.value_counts()

,count
Emotion,
anger,2000
joy,2000
fear,1937


In [69]:
df.shape

(5937, 2)

In [70]:
df.isnull().sum()

,0
Comment,0
Emotion,0


In [71]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5937 entries, 0 to 5936
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Comment  5937 non-null   object
 1   Emotion  5937 non-null   object
dtypes: object(2)
memory usage: 92.9+ KB


In [72]:
df1 = df.copy()

In [73]:
df1.duplicated().sum()

np.int64(0)

In [74]:
from nltk.corpus import stopwords
nltk.download('stopwords')
import string
from nltk.stem.snowball import SnowballStemmer
sb = SnowballStemmer('english')
import re

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [75]:
def transform_text(text):
  text = text.lower()
  text = nltk.word_tokenize(text)

  y = []
  for i in text:
    if i.isalnum():
      y.append(i)

  text = y[:]

  y.clear()

  for i in text:
     if i not in  stopwords.words('english')and i not in string.punctuation:
        y.append(i)

  text = y[:]
  y.clear()

  for i in text:
     y.append(sb.stem(i))

  return " ".join(y)

In [76]:
df1['Comment'] = df1['Comment'].apply(transform_text)

In [77]:
df1

,Comment,Emotion
0,serious hate one subject death feel reluct drop,fear
1,im full life feel appal,anger
2,sit write start dig feel think afraid accept p...,fear
3,ive realli angri r feel like idiot trust first...,joy
4,feel suspici one outsid like raptur happen someth,fear
...,...,...
5932,begun feel distress,fear
5933,left feel annoy angri think center stupid joke,anger
5934,ever get marri everyth readi offer got togeth ...,joy
5935,feel reluct appli want abl find compani know l...,fear



# GRU Model for Emotion Review Classification

This section adds a GRU (Gated Recurrent Unit) neural network model using TensorFlow/Keras
for emotion classification and prediction.


In [78]:

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense , Input , Bidirectional
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [79]:
texts = df1['Comment'].astype(str)
labels = df1['Emotion']


le = LabelEncoder()
labels_encoded = le.fit_transform(labels)

In [80]:
print("Original labels corresponding to encoded integers:", le.classes_)

Original labels corresponding to encoded integers: ['anger' 'fear' 'joy']


In [81]:
labels_encoded

array([1, 0, 1, ..., 2, 1, 0])

In [82]:
max_words = 10000
max_len = 100

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)


In [83]:
sequences = tokenizer.texts_to_sequences(texts)
X = pad_sequences(sequences, maxlen=max_len)


In [84]:
y = to_categorical(labels_encoded)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [85]:

# Build GRU Model
model = Sequential([
    Input(shape=(max_len,)),
    Embedding(input_dim=max_words, output_dim=64 ),
    Bidirectional(GRU(64)),
    Dense(32, activation='relu'),
    Dense(y.shape[1], activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 100, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        49,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 694,147 (2.65 MB)

 Trainable params: 694,147 (2.65 MB)

 Non-trainable params: 0 (0.00 B)

In [86]:

# Train GRU model
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)


Epoch 1/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 23s 147ms/step - accuracy: 0.4814 - loss: 1.0071 - val_accuracy: 0.4937 - val_loss: 0.8727
Epoch 2/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 16s 137ms/step - accuracy: 0.7607 - loss: 0.5357 - val_accuracy: 0.8905 - val_loss: 0.3261
Epoch 3/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 18s 153ms/step - accuracy: 0.9681 - loss: 0.1100 - val_accuracy: 0.9179 - val_loss: 0.2640
Epoch 4/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 17s 146ms/step - accuracy: 0.9874 - loss: 0.0351 - val_accuracy: 0.9305 - val_loss: 0.2093
Epoch 5/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 20s 141ms/step - accuracy: 0.9945 - loss: 0.0191 - val_accuracy: 0.9347 - val_loss: 0.1933
Epoch 6/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 17s 142ms/step - accuracy: 0.9971 - loss: 0.0102 - val_accuracy: 0.9358 - val_loss: 0.1955
Epoch 7/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 20s 142ms/step - accuracy: 0.9971 - loss: 0.0077 - val_accuracy: 0.9368 - val_loss: 0.2291
Epoch 8/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 17s 142ms/step - accuracy: 0.9989 - loss: 0

In [87]:

# Evaluate model
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", accuracy)


38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step - accuracy: 0.9453 - loss: 0.1971
Test Accuracy: 0.945286214351654


In [90]:

# Predict emotion for custom text
sample_text = "I am feeling very sad"

sample_seq = tokenizer.texts_to_sequences([sample_text])
sample_pad = pad_sequences(sample_seq, maxlen=max_len)

prediction = model.predict(sample_pad)
predicted_class = np.argmax(prediction)

print("Input Text:", sample_text)
print("Predicted Emotion Class:", le.inverse_transform([predicted_class])[0])


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step
Input Text: I am feeling very sad
Predicted Emotion Class: anger
